# PriceSense — Amazon Product Analysis and Recommendation System

**DSML Mini-Project**  
College Data Science and Machine Learning Project

---

## 1. Project Introduction

PriceSense is an end-to-end data science project that analyzes Amazon India product data to:

1. **Understand** product pricing, discount, and rating patterns
2. **Segment** products into meaningful groups using K-Means clustering
3. **Recommend** products based on user preferences using a simple rule-based system

**Dataset**: Amazon India products (1,465 raw rows → 1,351 unique products after cleaning)  
**Key Variables**: actual_price, discounted_price, discount_percentage, rating, rating_count  
**Categories**: 9 main categories (Electronics, Computers&Accessories, Home&Kitchen, etc.)  
**Only ML Algorithm**: K-Means Clustering (per DSML syllabus constraints)  
**Recommendation**: Rule-based, transparent, no collaborative filtering or embeddings

---

## 2. Objective

This notebook demonstrates the complete data science pipeline:

- **Data Preprocessing**: Handle Indian currency format (₹, commas), missing values, duplicates
- **Feature Engineering**: Create 12 derived features including log transforms
- **Statistical Analysis**: Central tendency, dispersion, distribution shape, correlation, covariance, pivot tables
- **Exploratory Data Analysis**: 36 visualizations (histograms, box plots, heatmaps, scatter plots)
- **K-Means Clustering**: 3 segments (Mainstream, Budget/Deep-Discount, Premium)
- **Cluster Analysis**: Business interpretation with category & discount band composition
- **Product Recommendation**: Rule-based filtering by category, price, rating, discount

All code is modularized in `src/` for reusability and viva-ready explanations.

---

## 3. Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Add src to path for importing modules
import sys
sys.path.append('../src')

print("Libraries imported successfully")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

## 4. Load Dataset

In [ ]:
# Load raw dataset
raw_df = pd.read_csv('../data/amazon.csv', dtype=str)
print(f"Raw dataset shape: {raw_df.shape}")
print(f"Columns: {list(raw_df.columns)}")
raw_df.head(3)

## 5. Dataset Inspection

In [ ]:
# Basic inspection
print("=== SHAPE ===")
print(f"Rows: {raw_df.shape[0]}, Columns: {raw_df.shape[1]}")

print("\n=== DTYPES ===")
print(raw_df.dtypes)

print("\n=== MISSING VALUES ===")
print(raw_df.isnull().sum())

print("\n=== SAMPLE VALUES ===")
for col in ['actual_price', 'discounted_price', 'discount_percentage', 'rating', 'rating_count', 'category']:
    if col in raw_df.columns:
        print(f"{col}: {raw_df[col].iloc[0]}")

## 6. Data Preprocessing

Using the reusable preprocessing module from `src/preprocessing.py`.

In [ ]:
# Import and run preprocessing pipeline
from preprocessing import preprocess_pipeline

df = preprocess_pipeline('../data/amazon.csv', save_path='../outputs/cleaned_data.csv')

print(f"\nCleaned dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

## 7. Data Cleaning

The preprocessing pipeline handles:

1. **Price cleaning**: `₹1,899` → `1899.0` (remove ₹ and commas)
2. **Discount cleaning**: `64%` → `64.0` (remove %)
3. **Rating cleaning**: Convert to float, validate 0-5 range
4. **Rating count cleaning**: `24,269` → `24269.0` (remove commas), **median imputation** for 2 missing values
5. **Duplicate handling**: 114 rows with repeated product_ids removed (kept first)
6. **Category parsing**: Pipe-separated hierarchy → main_category, sub_category, level_3-5

In [ ]:
# Verify cleaning results
print("=== MISSING VALUES AFTER CLEANING ===")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n=== NUMERIC COLUMNS STATS ===")
numeric_cols = ['actual_price', 'discounted_price', 'discount_percentage', 'rating', 'rating_count']
print(df[numeric_cols].describe().round(2))

print("\n=== CATEGORIES ===")
print(df['main_category'].value_counts())

## 8. Data Transformation

Key transformations applied:

- **Currency**: String → Float (Indian format handling)
- **Percentage**: String with % → Float (0-100 scale)
- **Rating count**: String with commas → Float, median imputation
- **Categories**: Hierarchical string → 5 level columns + depth

In [ ]:
# Show transformation examples
sample = df[['product_name', 'actual_price', 'discounted_price', 'discount_percentage', 'rating', 'rating_count', 'main_category']].head(5)
print(sample.to_string())

## 9. Feature Engineering

12 derived features created in the preprocessing pipeline:

In [ ]:
# Show engineered features
engineered = [
    'price_savings', 'discount_depth', 'price_ratio',
    'has_high_rating', 'is_heavily_discounted', 'is_budget', 'is_premium',
    'log_rating_count', 'log_discounted_price', 'log_actual_price',
    'review_count_parsed', 'avg_review_length'
]

print("=== ENGINEERED FEATURES ===")
for feat in engineered:
    if feat in df.columns:
        print(f"{feat}: {df[feat].dtype} | range: [{df[feat].min():.2f}, {df[feat].max():.2f}]")
    else:
        print(f"{feat}: NOT FOUND")

## 10. Statistical Analysis

Using `src/descriptive_stats.py` for comprehensive statistical analysis.

In [ ]:
# Run statistical analysis
from descriptive_stats import generate_statistical_report, interpret_skewness, interpret_kurtosis

key_vars = ["actual_price", "discounted_price", "discount_percentage", "rating", "rating_count"]
report = generate_statistical_report(df, key_vars)

# Summary statistics
summary = report["summary"]
print("=== COMPREHENSIVE SUMMARY STATISTICS ===")
print(summary.to_string())

# Interpretation
print("\n=== DISTRIBUTION INTERPRETATION ===")
for col in summary.index:
    skew = summary.loc[col, "Skewness"]
    kurt = summary.loc[col, "Kurtosis"]
    print(f"{col}: {interpret_skewness(skew)} (skew={skew:.2f}), {interpret_kurtosis(kurt)} (kurt={kurt:.2f})")

### Correlation Matrix (Pearson)

In [ ]:
corr = report["correlation"]
print("=== PEARSON CORRELATION MATRIX ===")
print(corr.round(4).to_string())

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, square=True,
            cbar_kws={'label': 'Pearson Correlation'})
plt.title('Correlation Matrix - Key Variables')
plt.tight_layout()
plt.show()

### Covariance Matrix

In [ ]:
cov = report["covariance"]
print("=== COVARIANCE MATRIX ===")
print(cov.round(2).to_string())

print("\nNote: Covariance values depend on variable scale. Use correlation for comparison.")

### Grouped Statistics by Category

In [ ]:
group_cat = report["group_by_category"]
mean_cols = [c for c in group_cat.columns if c.endswith('_mean')]
print("=== MEAN VALUES BY MAIN CATEGORY ===")
print(group_cat[['main_category'] + mean_cols].to_string(index=False))

### Grouped Statistics by Discount Band

In [ ]:
group_disc = report["group_by_discount_band"]
mean_cols = [c for c in group_disc.columns if c.endswith('_mean')]
print("=== MEAN VALUES BY DISCOUNT BAND ===")
print(group_disc[['discount_band'] + mean_cols].to_string(index=False))

## 11. Exploratory Data Analysis

Using `src/eda.py` for comprehensive visualizations. 36 charts generated and saved to `outputs/charts/`.

## 12. Distribution Analysis

In [ ]:
# Display distribution charts (generated by eda.py)
import matplotlib.image as mpimg

chart_files = [
    "../outputs/charts/actual_price_distribution.png",
    "../outputs/charts/discounted_price_distribution.png",
    "../outputs/charts/discount_percentage_distribution.png",
    "../outputs/charts/rating_distribution.png",
    "../outputs/charts/rating_count_distribution.png"
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, chart_file in enumerate(chart_files):
    img = mpimg.imread(chart_file)
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(chart_file.split('/')[-1].replace('_distribution.png', '').replace('_', ' ').title(), fontsize=12)

axes[-1].set_visible(False)
plt.suptitle('Distribution Analysis - Histograms with KDE', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 13. Correlation and Covariance

In [ ]:
# Display correlation heatmaps
corr_files = [
    "../outputs/charts/correlation_heatmap_pearson.png",
    "../outputs/charts/correlation_heatmap_spearman.png",
    "../outputs/charts/correlation_with_rating_pearson.png",
    "../outputs/charts/correlation_with_discounted_price_pearson.png"
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, corr_file in enumerate(corr_files):
    img = mpimg.imread(corr_file)
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(corr_file.split('/')[-1].replace('.png', '').replace('_', ' ').title(), fontsize=12)

plt.suptitle('Correlation Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 14. Pivot Tables

In [ ]:
# Display pivot table heatmaps
pivot_files = [
    "../outputs/charts/pivot_main_category_discount_band_count.png",
    "../outputs/charts/pivot_main_category_discount_band_rating_mean.png",
    "../outputs/charts/pivot_main_category_discount_band_discounted_price_mean.png",
    "../outputs/charts/pivot_main_category_discount_band_discount_percentage_mean.png"
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, pivot_file in enumerate(pivot_files):
    img = mpimg.imread(pivot_file)
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(pivot_file.split('/')[-1].replace('.png', '').replace('pivot_main_category_discount_band_', '').replace('_', ' ').title(), fontsize=11)

plt.suptitle('Pivot Tables: Category × Discount Band', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 15. K-Means Clustering

Only ML algorithm used (per DSML syllabus). Implemented in `src/clustering.py`.

In [ ]:
# Load cluster results
cluster_df = pd.read_csv('../outputs/cluster_results.csv')
print(f"Cluster results shape: {cluster_df.shape}")
print(f"Columns: {list(cluster_df.columns)}")

# Cluster sizes
print("\n=== CLUSTER SIZES ===")
print(cluster_df['cluster'].value_counts().sort_index())

### Feature Selection for Clustering

In [ ]:
# Feature selection rationale
features_rationale = {
    "discounted_price": "Price customer actually pays - primary purchase decision factor",
    "discount_percentage": "Discount depth - indicates pricing strategy and value perception",
    "rating": "Product quality/satisfaction signal - key trust indicator",
    "rating_count": "Review volume - proxy for popularity and market presence",
    "log_rating_count": "Log-transformed rating count - handles extreme skew (max 426K)",
    "log_discounted_price": "Log-transformed price - handles right-skew (max ₹78K)"
}

print("=== FEATURES SELECTED FOR CLUSTERING ===")
for feat, rationale in features_rationale.items():
    print(f"  • {feat}: {rationale}")

print("\n=== EXCLUDED ===")
print("  • actual_price: Highly correlated with discounted_price (r=0.96)")
print("  • price_savings, discount_depth, price_ratio: Derived from above")
print("  • review_count_parsed, avg_review_length: Redundant with rating/rating_count")

## 16. Elbow Method

In [ ]:
# Display Elbow Method chart
elbow_img = mpimg.imread("../outputs/charts/elbow_method.png")
plt.figure(figsize=(14, 6))
plt.imshow(elbow_img)
plt.axis('off')
plt.title('Elbow Method & Silhouette Analysis', fontsize=16, fontweight='bold')
plt.show()

print("=== ELBOW METHOD RESULTS ===")
elbow_data = {
    'K': [2, 3, 4, 5, 6, 7, 8, 9, 10],
    'Inertia': [6483.67, 5243.38, 4318.21, 3693.07, 3299.61, 2942.85, 2725.19, 2512.65, 2350.26],
    'Silhouette': [0.2118, 0.2188, 0.2317, 0.2286, 0.2225, 0.2349, 0.2305, 0.2280, 0.2220]
}
elbow_df = pd.DataFrame(elbow_data)
print(elbow_df.to_string(index=False))

### K Selection Rationale

- **Elbow method** suggests K=3 (rate of inertia decrease slows significantly)
- **Silhouette analysis** suggests K=7 (highest score: 0.2349)
- **Decision**: K=3 chosen — elbow method preferred for simpler, more interpretable model; silhouette difference at K=3 is not significant enough to justify added complexity

## 17. Cluster Analysis

In [ ]:
# Cluster statistics
key_vars = ["actual_price", "discounted_price", "discount_percentage", "rating", "rating_count", "price_savings"]
cluster_stats = cluster_df.groupby('cluster')[key_vars].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2)
print(cluster_stats.to_string())

### Cluster Interpretation (Based on Actual Statistics)

In [ ]:
# Load interpretations
interp_df = pd.read_csv('../outputs/cluster_interpretations.csv')
print(interp_df.to_string(index=False))

### Cluster Summary

In [ ]:
cluster_summary = pd.DataFrame({
    'Cluster': [0, 1, 2],
    'Label': ['Mainstream', 'Budget/Deep-Discount', 'Premium'],
    'Size': [646, 555, 150],
    '%': [47.8, 41.1, 11.1],
    'Avg_Price': [1461, 1023, 19687],
    'Avg_Discount%': [38.9, 59.8, 31.6],
    'Avg_Rating': [4.20, 3.94, 4.18],
    'Avg_Reviews': [31196, 2780, 14108]
})
print(cluster_summary.to_string(index=False))

### Cluster Composition by Category

In [ ]:
# Category composition
cat_comp = pd.read_csv('../outputs/cluster_category_pct.csv', index_col=0)
print("=== CLUSTER × CATEGORY (%) ===")
print(cat_comp.round(1).to_string())

# Discount band composition
disc_comp = pd.read_csv('../outputs/cluster_discount_band_pct.csv', index_col=0)
print("\n=== CLUSTER × DISCOUNT BAND (%) ===")
print(disc_comp.round(1).to_string())

## 18. Cluster Visualization

In [ ]:
# PCA Visualization
pca_img = mpimg.imread("../outputs/charts/clusters_pca.png")
plt.figure(figsize=(12, 9))
plt.imshow(pca_img)
plt.axis('off')
plt.title('K-Means Clusters via PCA (2D)', fontsize=16, fontweight='bold')
plt.show()

print("PCA explains 60.3% variance: PC1=34.3% (price), PC2=26.0% (rating count)")

In [ ]:
# Cluster profiles
profiles_img = mpimg.imread("../outputs/charts/cluster_profiles.png")
plt.figure(figsize=(12, 7))
plt.imshow(profiles_img)
plt.axis('off')
plt.title('Cluster Profiles (Normalized Feature Means)', fontsize=16, fontweight='bold')
plt.show()

In [ ]:
# Cluster sizes
sizes_img = mpimg.imread("../outputs/charts/cluster_sizes.png")
plt.figure(figsize=(10, 6))
plt.imshow(sizes_img)
plt.axis('off')
plt.title('Cluster Size Distribution', fontsize=16, fontweight='bold')
plt.show()

## 19. Product Recommendation

Simple, transparent, rule-based recommendation system implemented in `src/recommendation.py`.  
**No collaborative filtering, cosine similarity, embeddings, NLP, or neural networks.**

In [ ]:
# Import recommendation function
from recommendation import recommend_products, recommend_products_interactive

print("=== RECOMMENDATION SYSTEM LOGIC ===")
print(""
1. Input Validation: Validates category, max_price, min_rating, min_discount
2. Filtering Pipeline: Sequential filter by category → price → rating → discount
3. Ranking: Composite score with interpretable weights:
   - Rating (40%): Higher rating = better quality signal
   - Review count (30%, log-normalized): More reviews = more trustworthy
   - Discount % (20%): Higher discount = better value
   - Cluster boost (10%): Prefers Cluster 0 (Mainstream) for balanced value
4. Output: Top N products with key fields (product_id, name, category, prices, rating, cluster)
"")

## 20. Example Recommendations

In [ ]:
# Example 1: Budget Electronics
print("="*70)
print("EXAMPLE 1: Budget Electronics (under ₹1000, rating ≥ 4.0)")
print("="*70)
recommend_products_interactive(df, category='Electronics', max_price=1000, min_rating=4.0, n_recommendations=5)

In [ ]:
# Example 2: Home & Kitchen High Discount
print("="*70)
print("EXAMPLE 2: Home & Kitchen High Discount (≥50%, rating ≥ 4.0)")
print("="*70)
recommend_products_interactive(df, category='Home&Kitchen', min_discount=50, min_rating=4.0, n_recommendations=5)

In [ ]:
# Example 3: Premium Products
print("="*70)
print("EXAMPLE 3: Premium Products (₹5000-20000, rating ≥ 4.2)")
print("="*70)
recommend_products_interactive(df, max_price=20000, min_rating=4.2, n_recommendations=5)

In [ ]:
# Example 4: Top Rated Overall
print("="*70)
print("EXAMPLE 4: Top Rated Overall (no filters)")
print("="*70)
recommend_products_interactive(df, n_recommendations=5)

In [ ]:
# Example 5: Custom - Budget Home & Kitchen
print("="*70)
print("EXAMPLE 5: Budget Home & Kitchen (under ₹500, rating ≥ 4.0)")
print("="*70)
recommend_products_interactive(df, category='Home&Kitchen', max_price=500, min_rating=4.0, n_recommendations=5)

## 21. Conclusion

### Summary of Findings

**1. Data Quality & Preprocessing**
- Successfully cleaned 1,465 raw rows → 1,351 unique products (114 duplicates removed)
- Handled Indian currency format (₹, commas), percentage strings, and missing values
- 2 missing `rating_count` values imputed with **median** (robust to extreme skew)
- 12 engineered features created including log transforms for skewed variables

**2. Statistical Analysis**
- **Prices** heavily right-skewed (skew > 4): most products affordable (<₹5K), few premium outliers
- **Discounts** near-symmetric (median 49%): well-distributed across 0-94%
- **Ratings** left-skewed (median 4.1): 75% products rated ≥4.1
- **Review counts** extreme right-skew (max 426K): few products dominate reviews
- **Key correlation**: actual_price ↔ discounted_price (r=0.96), discount% ↔ rating (r=-0.16)

**3. Exploratory Data Analysis**
- 36 visualizations covering distributions, box plots, correlations, category analysis, pivot tables
- **Electronics** dominates: highest prices (₹10.4K avg), most products (490)
- **Sweet spot**: 50-70% discount band has best rating (4.13) with good savings
- **OfficeProducts**: lowest discount (12.4%), highest rating (4.31)

**4. K-Means Clustering (K=3)**
- **Features**: 6 selected (discounted_price, discount%, rating, log_rating_count, log_price) — justified
- **Scaling**: StandardScaler (required for Euclidean distance)
- **K selection**: Elbow at K=3 (silhouette=0.219), chosen over K=7 (silhouette=0.235) for interpretability
- **3 Segments identified**:
  - **Mainstream (47.8%)**: ₹1,461 avg, 38.9% discount, 4.20 rating, 31K reviews
  - **Budget/Deep-Discount (41.1%)**: ₹1,023 avg, 59.8% discount, 3.94 rating, 2.8K reviews
  - **Premium (11.1%)**: ₹19,687 avg, 31.6% discount, 4.18 rating, 14K reviews

**5. Cluster Analysis**
- **Mainstream**: Home&Kitchen, Computers, Electronics; spread across discount bands
- **Budget**: Heavy in 50-100% discount bands; quality trade-offs
- **Premium**: 81% Electronics; concentrated in 10-30% discount; brand strength
- **Strategic implications** documented for each segment

**6. Rule-Based Recommendation**
- 4 parameters: Category, Max Price, Min Rating, Min Discount
- Sequential filtering + composite score ranking
- Explainable, transparent, syllabus-compliant

### Project Compliance (DSML Syllabus)

| Requirement | Status |
|-------------|--------|
| Data loading, inspection, preprocessing | ✅ |
| Missing value handling (median for skew) | ✅ |
| Duplicate handling (exact vs repeated IDs) | ✅ |
| Feature transformation & engineering | ✅ (12 features) |
| Statistical analysis (central tendency, dispersion, distribution, percentiles) | ✅ |
| Correlation & covariance | ✅ |
| Grouping, aggregation, pivot tables | ✅ |
| EDA visualizations (histograms, box plots, heatmaps, scatter) | ✅ (36 charts) |
| **K-Means Clustering (ONLY ML algorithm)** | ✅ |
| Elbow Method & Silhouette for K selection | ✅ |
| Cluster analysis & business interpretation | ✅ |
| Rule-based recommendation system | ✅ |

### Prohibited Techniques (Not Used)

❌ Future price prediction / time series (ARIMA, Prophet, LSTM)  
❌ XGBoost, LightGBM, CatBoost, Gradient Boosting  
❌ Deep learning / neural networks  
❌ NLP / sentiment analysis  
❌ Collaborative filtering / matrix factorization  
❌ Web scraping / live APIs  

### Viva-Ready Key Points

1. **Why median for rating_count?** Extreme right-skew (max 426K, skew=5.69), median robust to outliers
2. **Why keep-first for duplicate product_ids?** Each row = same product with aggregated reviews in comma-separated strings
3. **Why StandardScaler?** K-Means uses Euclidean distance; features have vastly different scales
4. **Why K=3?** Elbow method primary; silhouette at K=3 (0.219) close to K=7 (0.235); simpler model preferred
5. **Why log transforms?** Handle extreme skew in rating_count and price for clustering
6. **Why no collaborative filtering?** Prohibited by syllabus; rule-based is explainable and privacy-compliant
7. **PCA only for visualization** — not primary ML algorithm (syllabus requires K-Means as only ML)

---

**Project Complete** ✅  
All phases implemented, tested, and documented.